# 01 – Regime-Conditioned Alpha Setup


V1 tested whether a simple Bayesian fair value model could detect exploitable mispricing in Polymarket prediction markets.

The result was negative: Bayesian mispricing alone did not produce robust trading alpha.

V2 shifted the focus from alpha generation to market efficiency.

The key finding was that prediction markets are not homogeneous. Instead, they appear to organize into two information-processing regimes:

1. **Information Processing Markets**
2. **Anchored / Noisy Markets**

These regimes showed dramatically different forecast accuracy.

V3 starts from that discovery.

The central question is no longer: Are prediction markets efficient?

but rather: Can information-processing regimes help isolate where tradable alpha may exist?

---

## Research Question

Does Bayesian mispricing become more useful when conditioned on market regime?

---

## Core Hypothesis

A global Bayesian mispricing signal may fail because it treats all markets as identical.

However, mispricing may behave differently across information regimes.

In particular:

* Information Processing Markets may quickly eliminate mispricing.
* Anchored / Noisy Markets may update slowly and preserve exploitable inefficiencies.

Therefore, alpha may not exist globally, but may exist conditionally.

---

## Notebook Objective

This notebook prepares the V3 modeling dataset by merging:

1. Bayesian fair value outputs from V1
2. Information-dynamics features from V2
3. Market-state cluster labels from V2
4. Final outcomes and market probabilities

The resulting dataset will allow us to test whether mispricing performance differs across market regimes.

---


## Regime Definitions

### Information Processing Markets

Markets characterized by:

* High Drawdown
* High Reversals
* High Probability Range
* Low Entropy

These markets tend to incorporate information efficiently and converge toward accurate forecasts.

---

### Regime 1  Anchored / Noisy Markets

Markets characterized by:

* Low Drawdown
* Low Reversals
* Low Probability Range
* High Entropy

These markets tend to update slowly, remain noisy, and exhibit larger forecast errors.

---

## Main Test Prepared by This Notebook

The final dataset will support the following analysis:

```text
Bayesian Mispricing
        ↓
Condition on Market Regime
        ↓
Signal Quality by Regime
```

The next notebooks will evaluate whether BUY/SELL signals perform better inside specific market regimes.

---


In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

fair_value = pd.read_csv("../data/processed/market_dataset_with_fair_value.csv")
df_model_v2 = pd.read_csv("../data/processed/df_model_v2.csv")

fair_cols = [
    "market_id",
    "question",
    "outcome",
    "final_probability",
    "bayesian_fair_probability",
    "mispricing"
]

features = [
    "realized_volatility",
    "probability_range",
    "trend",
    "max_drawdown",
    "reversals",
    "shannon_entropy",
    "skewness",
    "kurtosis",
    "autocorrelation"
]


fair_value_base = fair_value[fair_cols].copy()
info_base = df_model_v2[["market_id", "abs_surprise"] + features].copy()
regime_alpha = fair_value_base.merge(info_base,on="market_id", how="left")

In [5]:
regime_alpha.shape

(43, 16)

In [11]:
regime_alpha.isna().sum().sort_values(ascending=False)

autocorrelation              4
shannon_entropy              4
market_id                    0
outcome                      0
question                     0
final_probability            0
bayesian_fair_probability    0
realized_volatility          0
probability_range            0
mispricing                   0
abs_surprise                 0
max_drawdown                 0
trend                        0
skewness                     0
reversals                    0
kurtosis                     0
cluster                      0
market_regime                0
abs_mispricing               0
dtype: int64

In [ ]:
X = regime_alpha[features]
preprocess = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
X_scaled = preprocess.fit_transform(X)

In [7]:
kmeans = KMeans(n_clusters=2,random_state=42,n_init=20)
regime_alpha["cluster"] = kmeans.fit_predict(X_scaled)

In [8]:
cluster_summary = regime_alpha.groupby("cluster")[["max_drawdown", "reversals", "probability_range", "shannon_entropy", "abs_surprise"]
].mean()
cluster_summary

,max_drawdown,reversals,probability_range,shannon_entropy,abs_surprise
cluster,,,,,
0,0.870148,702.843750,0.617969,0.181319,0.040422
1,0.447587,96.727273,0.218455,0.568352,0.340091


In [9]:
regime_map = {0: "Information Processing",
              1: "Anchored / Noisy"}

regime_alpha["market_regime"] = regime_alpha["cluster"].map(regime_map)

In [10]:
regime_alpha["abs_mispricing"] = regime_alpha["mispricing"].abs()
regime_alpha[[ "market_id",
              "question",
              "mispricing",
              "abs_mispricing",
              "abs_surprise"]].head()

,market_id,question,mispricing,abs_mispricing,abs_surprise
0,248594,Will Hunter Biden be federally indicted by May...,0.018462,0.018462,0.020
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,0.050000,0.050000,0.450
2,250474,Did US GDP grow 2.5% or more in Q1 2023?,-0.026667,0.026667,0.360
3,251025,Will Ron DeSantis file to run for president by...,-0.095000,0.095000,0.005
4,251027,USD/TRY (Turkish Lira) above 19.75 on May 22?,-0.158333,0.158333,0.175


In [12]:
regime_alpha.to_csv("../data/processed/regime_alpha_dataset.csv",index=False)

# Is Bayesian mispricing different by regime?

In [15]:
regime_alpha.groupby("market_regime")["abs_mispricing"].agg(["count", "mean", "median", "std", "min", "max"])

,count,mean,median,std,min,max
market_regime,,,,,,
Anchored / Noisy,11,0.075224,0.045000,0.074876,0.021538,0.265
Information Processing,32,0.056990,0.037962,0.047882,0.011538,0.265


In [ ]:
regime_alpha.groupby("market_regime")["mispricing"].agg(["count", "mean", "median", "std", "min", "max"])

,count,mean,median,std,min,max
market_regime,,,,,,
Anchored / Noisy,11,0.011308,0.028462,0.108121,-0.158333,0.265
Information Processing,32,-0.000973,0.030962,0.075129,-0.108333,0.265


In [ ]:
regime_alpha["signal"] = np.where(regime_alpha["mispricing"] > 0, "BUY", "SELL")

regime_alpha.groupby(["market_regime", "signal"])["outcome"].agg(["count", "mean"]) 

count      mean
market_regime          signal                 
Anchored / Noisy       BUY         6  0.500000
                       SELL        5  0.200000
Information Processing BUY        20  0.000000
                       SELL       12  0.833333